# Bayesian Hierarchical Logistic Regression for Malaria Risk

Python workflow for the Gambia malaria random-effects logistic regression project.

This notebook loads the data, fits a baseline logistic regression model, fits a Bayesian hierarchical logistic regression model, and creates spatial visualizations of village-level malaria risk.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from data_prep import load_gambia_data, make_model_matrices
from baseline_logistic import fit_baseline_logistic, fit_train_test_logistic
from bayesian_random_effects import fit_bayesian_random_effects
from visualization import (
    plot_baseline_coefficients,
    plot_gambia_village_map,
    plot_village_effect_maps,
    save_trace_plot,
)

DATA_PATH = PROJECT_ROOT / "data" / "gambia.csv"
BORDER_PATH = PROJECT_ROOT / "data" / "gambia_borders.csv"
FIGURES_DIR = PROJECT_ROOT / "figures"
RESULTS_DIR = PROJECT_ROOT / "results"

FIGURES_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

## Load Data

The dataset contains child-level malaria infection outcomes, covariates, and spatial coordinates. Village identifiers are constructed from shared `(x, y)` coordinate pairs.

In [ ]:
df = load_gambia_data(DATA_PATH)
df.head()

In [ ]:
X, y, village, feature_names = make_model_matrices(df)

print("Observations:", len(y))
print("Malaria-positive cases:", y.sum())
print("Malaria prevalence:", round(y.mean(), 3))
print("Villages:", village.max() + 1)

## Plot Village Locations

This plot recreates the original R-style village location map using the Gambia border file and village coordinates.

In [ ]:
plot_gambia_village_map(
    df,
    BORDER_PATH,
    FIGURES_DIR / "gambia_village_locations.png",
)

## Baseline Logistic Regression

The baseline model estimates malaria infection probability using observed covariates only. This provides a simple benchmark before adding village-level random effects.

In [ ]:
baseline = fit_baseline_logistic(X, y, feature_names)
baseline["summary"]

In [ ]:
baseline["metrics"]

In [ ]:
test_baseline = fit_train_test_logistic(X, y, feature_names)
test_baseline["metrics"]

In [ ]:
baseline["summary"].to_csv(
    RESULTS_DIR / "baseline_logistic_coefficients.csv",
    index=False,
)

plot_baseline_coefficients(
    baseline["summary"],
    FIGURES_DIR / "baseline_logistic_coefficients.png",
)

## Bayesian Hierarchical Logistic Regression

The Bayesian model adds village-level random intercepts. Positive coefficients increase malaria log-odds. Positive village effects indicate villages with higher malaria risk than expected after controlling for observed covariates.

In [ ]:
bayes = fit_bayesian_random_effects(
    X,
    y,
    village,
    feature_names,
    draws=1000,
    tune=1000,
    chains=4,
)

bayes["coef_summary"]

In [ ]:
bayes["village_effects"].head()

In [ ]:
bayes["coef_summary"].to_csv(
    RESULTS_DIR / "bayesian_coefficient_summary.csv",
)

bayes["village_effects"].to_csv(
    RESULTS_DIR / "village_random_effects.csv",
    index=False,
)

## Spatial Random-Effects Maps

The first map shows the posterior mean random effect for each village. The second map shows the posterior probability that each village's random effect is positive.

In [ ]:
village_coords = df[["village_id", "x", "y"]].drop_duplicates()
village_plot_df = village_coords.merge(
    bayes["village_effects"],
    on="village_id",
)

plot_village_effect_maps(
    village_plot_df,
    BORDER_PATH,
    FIGURES_DIR / "village_effects",
)

## MCMC Diagnostics

Trace plots are saved for key Bayesian model parameters.

In [ ]:
save_trace_plot(
    bayes["trace"],
    FIGURES_DIR / "bayesian_trace_plot.png",
)

## Interpretation

Positive regression coefficients indicate higher malaria log-odds. Negative coefficients indicate lower malaria log-odds.

Positive village random effects indicate villages with higher malaria risk than expected after accounting for age, bed net use, treatment status, vegetation, and health center access.